# Developer Burnout Analysis Pipeline

## A Complete ML Workflow from Data to Deployment

This notebook demonstrates a complete, production-grade machine learning pipeline using real-world developer burnout data. You'll learn how to:

1. Load and explore data
2. Preprocess and prepare data for modeling
3. Engineer meaningful features
4. Split data properly (train/test with stratification)
5. Train multiple models
6. Evaluate and compare models
7. Track experiments with MLflow
8. Prepare models for deployment

**Key Learning Objectives:**
- Understand the complete ML lifecycle
- Learn best practices for data handling
- Discover how to compare multiple algorithms
- Master experiment tracking and reproducibility
- Prepare models for production serving

## What You'll Learn

This notebook covers the **complete pipeline** that real ML engineers use:

**Data Phase:**
- How to handle missing values intelligently
- Why normalization matters
- Categorical encoding techniques

**Modeling Phase:**
- Comparing different algorithms (Logistic Regression, Random Forest)
- Why stratified splitting is important
- Feature engineering for better predictions

**Evaluation Phase:**
- Multiple metrics beyond accuracy
- Confusion matrices and ROC curves
- Why AUC tells a different story than accuracy

**Production Phase:**
- Experiment tracking with MLflow
- Model reproducibility
- Making predictions at scale

## Section 0: Imports and Setup

Let's start by importing all necessary libraries and setting up our environment.

In [ ]:
# Standard library imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn utilities
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score

# Project modules
from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor
from src.data.splitter import DataSplitter
from src.features.engineer import FeatureEngineer
from src.models.trainer import ModelTrainer
from src.evaluation.metrics import EvaluationMetrics
from src.mlflow_integration.tracker import MLflowTracker

# Set random seed for reproducibility
np.random.seed(42)

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ All imports successful!")
print(f"✓ Project root: {project_root}")

## Section 1: Exploratory Data Analysis (EDA)

### Step 1: Load the Data

**Why this matters:** Before we build any models, we need to understand our data. What are we predicting? What features do we have? Are there any issues we need to handle?

The **Developer Burnout Dataset** contains metrics about software developers:
- **Target**: `burnout_level` (Low, Medium, High) - what we're predicting
- **Features**: work metrics, sleep, stress, exercise, etc.
- **Challenge**: Imbalanced classes and missing values

We use `DataLoader` to safely load and validate the data.

In [ ]:
# Load the sources using our DataLoader utility
data_path = project_root / 'sources' / 'raw' / '1_developer_burnout_dataset_7000.csv'

loader = DataLoader(str(data_path))
df_raw, info = loader.load_with_info()

print("Dataset Overview:")
print(f"Shape: {info['shape']} (rows × columns)")
print(f"\nColumns: {', '.join(info['columns'])}")
print(f"\nDataset loaded successfully!")
print(f"Sample of first 5 rows:")
df_raw.head()

### Understanding the Data

Let's examine the structure and identify any issues.

In [ ]:
# Display basic statistics
print("=" * 80)
print("DATA TYPES AND MISSING VALUES")
print("=" * 80)

data_info = pd.DataFrame({
    'Column': df_raw.columns,
    'Type': df_raw.dtypes.values,
    'Non-Null Count': df_raw.notna().sum().values,
    'Null Count': df_raw.isnull().sum().values,
    'Null %': (df_raw.isnull().sum().values / len(df_raw) * 100).round(2)
})

print(data_info.to_string(index=False))

print("\n" + "=" * 80)
print("STATISTICAL SUMMARY")
print("=" * 80)
df_raw.describe()

### Class Distribution: Understanding Imbalance

**Why this matters:** In classification problems, imbalanced classes (one class has many more samples than others) can bias our models. We use **stratified splitting** to ensure each train/test set has similar class proportions.

In [ ]:
# Check target variable distribution
print("\n" + "=" * 80)
print("TARGET VARIABLE DISTRIBUTION (Burnout Level)")
print("=" * 80)

class_dist = df_raw['burnout_level'].value_counts()
class_pct = df_raw['burnout_level'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({
    'Burnout Level': class_dist.index,
    'Count': class_dist.values,
    'Percentage': class_pct.values.round(2)
})

print(dist_df.to_string(index=False))

# Visualize class distribution
fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df_raw, x='burnout_level', ax=ax, palette='Set2')
ax.set_title('Distribution of Burnout Levels in Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Burnout Level', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
plt.tight_layout()
plt.show()

print("\n✓ Class distribution analyzed. Notice the imbalance - this is why stratification matters!")

## Section 2: Data Preprocessing

### Step 2: Prepare Data for Modeling

**Why preprocessing matters:**

1. **Handling Missing Values**: Some developers didn't report daily work hours. We fill these with the mean (average).
2. **Categorical Encoding**: Algorithms need numbers, not text. 'Low'/'Medium'/'High' → 0/1/2
3. **Normalization**: Features with different scales (sleep hours: 2-10, bugs per day: 0-20) confuse algorithms. We normalize to mean=0, std=1.

**The Pipeline Principle**: Fit preprocessing on training data only, then apply it to test data. This prevents "data leakage".

In [ ]:
# Make a copy for preprocessing
df = df_raw.copy()

# Identify column types
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = ['burnout_level']

# Remove target from numeric columns
numeric_cols = [col for col in numeric_cols if col not in categorical_cols]

print("Column Classification:")
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

# Initialize preprocessor
preprocessor = DataPreprocessor(
    df=df,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols
)

# Apply preprocessing pipeline
df_processed = preprocessor.fit_transform(strategy='mean')

print("\n✓ Preprocessing complete!")
print(f"Shape after preprocessing: {df_processed.shape}")
print(f"\nFirst few rows after preprocessing:")
df_processed.head()

### Verify Preprocessing Results

Let's check that our preprocessing worked correctly.

In [ ]:
print("=" * 80)
print("PREPROCESSING VERIFICATION")
print("=" * 80)

# Check for missing values
print("\n1. Missing Values After Preprocessing:")
missing_after = df_processed.isnull().sum()
print(f"   Total missing values: {missing_after.sum()}")
print(f"   ✓ All missing values handled!" if missing_after.sum() == 0 else "   ✗ Some missing values remain")

# Check normalization
print("\n2. Feature Normalization (Sample):")
print("   (Mean should be ~0, Std should be ~1 for normalized features)")
sample_numeric = numeric_cols[:3]
for col in sample_numeric:
    mean = df_processed[col].mean()
    std = df_processed[col].std()
    print(f"   {col}: mean={mean:.4f}, std={std:.4f}")

# Check encoding
print("\n3. Categorical Encoding:")
print(f"   burnout_level unique values: {df_processed['burnout_level'].unique()}")
print(f"   (0=Low, 1=Medium, 2=High)")

print("\n✓ Preprocessing verified!")

## Section 3: Feature Engineering

### Step 3: Create New Features

**Why feature engineering matters:**

Raw features are good, but *engineered* features are better. We create meaningful combinations:

1. **Interaction Features**: `work_hours × meetings` - long hours with many meetings means high stress
2. **Ratio Features**: `bugs ÷ commits` - quality metric, shows code reliability
3. **Polynomial Features**: `stress_level²` - captures non-linear relationships

These help algorithms discover patterns that raw features alone would miss.

In [ ]:
# Create copy for feature engineering
df_featured = df_processed.copy()

# Initialize feature engineer
engineer = FeatureEngineer(df_featured)

# Create ratio features (quality metrics)
# Higher bugs_per_day / commits_per_day = lower quality
df_featured = engineer.create_ratio_features('bugs_per_day', 'commits_per_day')

# Create interaction features (work intensity)
# High work hours + many meetings = extreme stress
df_featured = engineer.create_interaction_features(
    ['daily_work_hours', 'meetings_per_day']
)

# Create polynomial features (stress indicators)
# Stress level has non-linear impact on burnout
df_featured = engineer.create_polynomial_features(
    ['stress_level'],
    degree=2
)

print("=" * 80)
print("FEATURE ENGINEERING RESULTS")
print("=" * 80)
print(f"Original features: {len(numeric_cols)}")
print(f"New features created:")
print(f"  - Ratio features: 1 (bugs_div_commits)")
print(f"  - Interaction features: 1 (daily_work_hours_x_meetings_per_day)")
print(f"  - Polynomial features: 1 (stress_level_2)")
print(f"Total features after engineering: {len(df_featured.columns) - 1} (excluding target)")
print(f"\nNew column names:")
new_cols = [col for col in df_featured.columns if col not in df_processed.columns]
for col in new_cols:
    print(f"  - {col}")

## Section 4: Train/Test Splitting

### Step 4: Split Data Properly

**Why stratified splitting matters:**

Random splitting can accidentally put all 'High' burnout developers in training and none in testing. **Stratified splitting** ensures both sets have the same class proportions (e.g., 20% High, 40% Medium, 40% Low in both).

This prevents unrealistic evaluation metrics.

In [ ]:
# Separate features and target
X = df_featured.drop('burnout_level', axis=1)
y = df_featured['burnout_level']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Use stratified splitting
splitter = DataSplitter(
    df=df_featured,
    target_col='burnout_level',
    test_size=0.2,
    random_state=42
)

# Split with stratification
train_df, test_df = splitter.train_test_split(stratify=True)

print("\n" + "=" * 80)
print("SPLIT VERIFICATION (Stratification Works!)")
print("=" * 80)

# Compare class distributions
print("\nOriginal class distribution:")
print(df_featured['burnout_level'].value_counts(normalize=True).round(3))

print("\nTraining set class distribution:")
print(train_df['burnout_level'].value_counts(normalize=True).round(3))

print("\nTest set class distribution:")
print(test_df['burnout_level'].value_counts(normalize=True).round(3))

# Prepare for modeling
X_train = train_df.drop('burnout_level', axis=1)
y_train = train_df['burnout_level']

X_test = test_df.drop('burnout_level', axis=1)
y_test = test_df['burnout_level']

print(f"\n✓ Split complete!")
print(f"  Training set: {X_train.shape}")
print(f"  Test set: {X_test.shape}")

## Section 5: Model Training

### Step 5: Train and Compare Multiple Models

**Why we compare multiple algorithms:**

Different algorithms excel at different tasks:

- **Logistic Regression**: Fast, interpretable, works well with normalized features
- **Random Forest**: Handles non-linear patterns, feature importance, less sensitive to scaling

We train both and compare their performance to pick the best one.

In [ ]:
# Convert to numpy arrays for modeling
X_train_np = X_train.values
y_train_np = y_train.values

X_test_np = X_test.values
y_test_np = y_test.values

# Initialize trainer
trainer = ModelTrainer(cv=5)  # 5-fold cross-validation

# Define algorithms and hyperparameters
algorithms = ['logistic_regression', 'random_forest']

params_dict = {
    'logistic_regression': {
        'max_iter': 1000,
        'random_state': 42
    },
    'random_forest': {
        'n_estimators': 100,
        'max_depth': 10,
        'random_state': 42,
        'n_jobs': -1
    }
}

# Train all models
print("Training models...")
training_results = trainer.train_multiple(X_train_np, y_train_np, algorithms, params_dict)

print("\n" + "=" * 80)
print("TRAINING RESULTS")
print("=" * 80)

results_summary = []
for result in training_results:
    results_summary.append({
        'Algorithm': result['algorithm'],
        'Training Accuracy': f"{result['metrics']['accuracy']:.4f}",
        'CV Mean': f"{result['metrics']['cv_mean']:.4f}",
        'CV Std': f"{result['metrics']['cv_std']:.4f}"
    })

results_df = pd.DataFrame(results_summary)
print(results_df.to_string(index=False))

# Store for later use
trained_models = {r['algorithm']: r['model'] for r in training_results}

print(f"\n✓ Training complete! {len(trained_models)} models trained.")

## Section 6: Model Evaluation

### Step 6: Evaluate and Select the Best Model

**Why multiple metrics matter:**

Accuracy alone is misleading. Imagine a dataset where 99% of developers have low burnout:
- A model that *always* predicts "Low" gets 99% accuracy
- But it fails completely at identifying high-burnout developers

We use:
- **Precision**: Of cases we predict as High burnout, how many are actually High?
- **Recall**: Of actual High burnout cases, how many do we catch?
- **F1**: Balanced average of precision and recall
- **AUC-ROC**: Area under the receiver operating characteristic curve (best for comparing models)

In [ ]:
# Make predictions on test set
print("=" * 80)
print("TEST SET EVALUATION")
print("=" * 80)

evaluation_results = []

for algo_name, model in trained_models.items():
    # Get predictions
    y_pred = model.predict(X_test_np)
    y_proba = model.predict_proba(X_test_np)[:, 1]
    
    # Calculate metrics
    evaluator = EvaluationMetrics(y_test_np, y_proba, is_proba=True)
    metrics = evaluator.calculate_metrics()
    
    evaluation_results.append({
        'algorithm': algo_name,
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'metrics': metrics
    })
    
    print(f"\n{algo_name.upper()}")
    print("-" * 40)
    for metric, value in metrics.items():
        print(f"{metric.capitalize():12s}: {value:.4f}")

# Find best model
best_result = max(evaluation_results, key=lambda x: x['metrics']['f1'])
best_algo = best_result['algorithm']
best_model = best_result['model']

print(f"\n" + "=" * 80)
print(f"✓ Best model: {best_algo.upper()} (F1 Score: {best_result['metrics']['f1']:.4f})")
print("=" * 80)

### Detailed Evaluation: Confusion Matrix and Classification Report

In [ ]:
# Focus on best model
best_y_pred = best_result['y_pred']

# Get confusion matrix
cm = confusion_matrix(y_test_np, best_y_pred)

# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Low', 'Medium', 'High'],
    yticklabels=['Low', 'Medium', 'High'],
    ax=ax,
    cbar_kws={'label': 'Count'}
)
ax.set_title(f'Confusion Matrix - {best_algo.upper()}', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print("\nConfusion Matrix Interpretation:")
print("-" * 60)
print("Diagonal = Correct predictions (True Positives/Negatives)")
print("Off-diagonal = Incorrect predictions (False Positives/Negatives)")

# Get detailed report
evaluator = EvaluationMetrics(y_test_np, best_y_pred, is_proba=False)
report = evaluator.get_classification_report()
print("\n" + report)

### ROC Curve Visualization

**Why ROC curves matter:**

The ROC curve shows the trade-off between catching high-burnout cases (True Positive Rate) and false alarms (False Positive Rate). AUC (area under the curve) ranges from 0.5 (random guessing) to 1.0 (perfect prediction).

In [ ]:
# Plot ROC curves for all models
fig, ax = plt.subplots(figsize=(10, 6))

for result in evaluation_results:
    algo_name = result['algorithm']
    y_proba = result['y_proba']
    
    # Calculate ROC curve
    fpr, tpr, _ = roc_curve(y_test_np, y_proba)
    roc_auc = auc(fpr, tpr)
    
    ax.plot(
        fpr, tpr,
        label=f'{algo_name.upper()} (AUC = {roc_auc:.3f})',
        linewidth=2
    )

# Plot random classifier line
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.500)', linewidth=1)

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ ROC analysis complete!")

### MLflow Experiment Tracking

**Why experiment tracking matters:**

In production, you'll train thousands of models. MLflow tracks:
- Parameters (hyperparameters)
- Metrics (accuracy, F1, etc.)
- Models themselves
- Artifacts (plots, configs)

This lets you compare experiments and reproduce the best one.

In [ ]:
# Initialize MLflow tracker
tracker = MLflowTracker(
    experiment_name='developer_burnout_experiment',
    tracking_uri='file:./models/mlruns'
)

# Log all models
print("Logging models to MLflow...")
for result in evaluation_results:
    algo_name = result['algorithm']
    model = result['model']
    metrics = result['metrics']
    params = params_dict[algo_name]
    
    # Start run
    tracker.start_run(run_name=f"run_{algo_name}")
    
    # Log parameters
    tracker.log_params({
        'algorithm': algo_name,
        **params
    })
    
    # Log metrics
    tracker.log_metrics(metrics)
    
    # Log model
    tracker.log_model(model.model, model_format='sklearn')
    
    # End run
    tracker.end_run()
    
    print(f"✓ Logged {algo_name}")

print("\n✓ All models logged to MLflow!")
print(f"Tracking URI: ./models/mlruns")

## Section 7: Deployment Preparation

### Step 7: Prepare for Production

In production, the model needs to make predictions on new, unseen developer data. Let's show how this works.

In [ ]:
# Single prediction example
print("=" * 80)
print("SINGLE PREDICTION EXAMPLE")
print("=" * 80)

# Get a sample from test set
sample_idx = 0
sample_features = X_test_np[sample_idx:sample_idx+1]

# Make prediction
pred_class = best_model.predict(sample_features)[0]
pred_proba = best_model.predict_proba(sample_features)[0]

burnout_levels = ['Low', 'Medium', 'High']
predicted_level = burnout_levels[pred_class]

print(f"\nSample Developer Profile:")
sample_df = pd.DataFrame(
    sample_features,
    columns=X_test.columns
)
print(sample_df.T.to_string())

print(f"\nPredicted Burnout Level: {predicted_level}")
print(f"\nProbability Distribution:")
for level, prob in zip(burnout_levels, pred_proba):
    print(f"  {level:8s}: {prob:.4f} ({prob*100:6.2f}%)")

print(f"\n✓ Prediction for single developer complete!")

### Batch Predictions

In production, you'll often predict for many developers at once.

In [ ]:
# Batch predictions
print("=" * 80)
print("BATCH PREDICTIONS (First 10 developers)")
print("=" * 80)

# Predict on first 10 test samples
batch_features = X_test_np[:10]
batch_predictions = best_model.predict(batch_features)
batch_probabilities = best_model.predict_proba(batch_features)

# Create results dataframe
burnout_levels = ['Low', 'Medium', 'High']
batch_results = pd.DataFrame({
    'True Level': [burnout_levels[y] for y in y_test_np[:10]],
    'Predicted Level': [burnout_levels[y] for y in batch_predictions],
    'Low Prob': batch_probabilities[:, 0].round(4),
    'Medium Prob': batch_probabilities[:, 1].round(4),
    'High Prob': batch_probabilities[:, 2].round(4),
    'Correct': (batch_predictions == y_test_np[:10]).astype(int)
})

print(batch_results.to_string(index=False))

accuracy = batch_results['Correct'].mean()
print(f"\nBatch Accuracy (first 10): {accuracy:.2%}")
print(f"\n✓ Batch prediction complete!")

## Section 8: Summary and Key Learnings

### What We Accomplished

We completed a full ML pipeline from raw data to production-ready predictions:

1. **Loaded** 7,000 developer records with burnout metrics
2. **Explored** data to identify challenges (missing values, class imbalance)
3. **Preprocessed** by handling missing values, encoding categories, normalizing features
4. **Engineered** new features (ratios, interactions, polynomials) to improve predictions
5. **Split** data with stratification to ensure fair evaluation
6. **Trained** multiple algorithms and compared performance
7. **Evaluated** using multiple metrics (accuracy, precision, recall, F1, AUC)
8. **Tracked** experiments with MLflow for reproducibility
9. **Deployed** by making single and batch predictions

### Key Principles Learned

**Data Quality First**: Garbage in, garbage out. Preprocessing matters more than the algorithm.

**Understand the Problem**: Why are we building this? What decisions will it inform? (Here: identifying burned-out developers)

**Stratified Splitting**: Maintain class balance in train/test to avoid misleading metrics.

**Multiple Metrics**: Accuracy alone is insufficient. Use precision, recall, F1, and AUC depending on your use case.

**Experiment Tracking**: Keep detailed records of parameters, metrics, and models for reproducibility.

**Feature Engineering**: Domain knowledge beats raw features. Think about what signals matter.

### Next Steps

In a real project, you'd continue with:

1. **Hyperparameter Tuning**: Use grid search or Bayesian optimization to find optimal parameters
2. **Cross-Validation**: More thorough validation with k-fold splits
3. **Feature Selection**: Identify which features matter most
4. **Ensemble Methods**: Combine multiple models for better performance
5. **Model Deployment**: Package as API (FastAPI) or microservice
6. **Monitoring**: Track model performance in production and retrain when needed
7. **Fairness Analysis**: Ensure model doesn't discriminate unfairly

### Resources

- **scikit-learn**: https://scikit-learn.org - machine learning library
- **MLflow**: https://mlflow.org - experiment tracking
- **Pandas**: https://pandas.pydata.org - data manipulation
- **Matplotlib/Seaborn**: visualization libraries

### Questions to Explore

1. Why did Random Forest outperform Logistic Regression? (non-linear patterns in data)
2. What if we had more data? (more training examples = better generalization)
3. Which features matter most? (can use feature importance from Random Forest)
4. How to handle class imbalance better? (SMOTE, class weights, threshold adjustment)
5. Can we improve further? (ensemble methods, deeper hyperparameter tuning)

Happy learning!

In [ ]:
# Final summary statistics
print("\n" + "=" * 80)
print("PIPELINE EXECUTION SUMMARY")
print("=" * 80)

summary_stats = {
    'Total Records': len(df_raw),
    'Training Records': len(X_train),
    'Test Records': len(X_test),
    'Original Features': len(numeric_cols),
    'Engineered Features': len(X_test.columns),
    'Best Model': best_algo,
    'Best F1 Score': f"{best_result['metrics']['f1']:.4f}",
    'Best Accuracy': f"{best_result['metrics']['accuracy']:.4f}",
    'Best AUC': f"{best_result['metrics']['auc']:.4f}",
}

for key, value in summary_stats.items():
    print(f"{key:25s}: {value}")

print("\n" + "=" * 80)
print("✓ Pipeline execution complete! Model is ready for deployment.")
print("=" * 80)